In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install natasha

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 66.4 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=5e3986aeb051eec5501c70b9df43fb504a20a4c1a09c3f4be503416b720cbbce
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
  Created wheel for intervaltree: filename=intervaltree-3.1.0-py2.py3-none-any.whl size=26098 sha256=8483e4512bc21ce1ac3d3d02d19aed166eb362f964e41d8e8d426b7e61817ee1
  Stored in directory: /root/.cache/pip/wheels/65/c3/c3/238bf93c243597857edd94ddb0577faa74a8e16e9585896e83
Successfully built docopt intervaltree


In [3]:
from pathlib import Path
import pandas as pd
import networkx as nx
import re
import numpy as np
from scipy.spatial.distance import cosine
import os

from natasha import (
    NamesExtractor, PER, Doc, NewsEmbedding,
    NewsSyntaxParser, NewsMorphTagger,
    Segmenter, NewsNERTagger, MorphVocab
)

In [4]:
# Пути
DATA_PATH = Path('/content/drive/My Drive/SFU 4/ML/NER/Natasha_Spacy')
corpus_path = DATA_PATH / 'corpus'

In [5]:
# Инициализация Natasha
segmenter = Segmenter()
morph_vocab = MorphVocab()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)
syntax_parser = NewsSyntaxParser(emb)
ner_tagger = NewsNERTagger(emb)
names_extractor = NamesExtractor(morph_vocab)

In [6]:
# Функции работы с текстами
def open_all_texts(corpus_path):
    texts = []
    for filename in sorted(os.listdir(corpus_path)):
        if filename.endswith('.txt'):
            file_path = os.path.join(corpus_path, filename)
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
                texts.append((filename, content))
    return texts

def preprocess_texts(texts):
    cleaned = []
    for filename, text in texts:
        text = re.sub(r'(?!\n)\s+', ' ', text).strip()
        text = re.sub(r'\n+', ' ; \n', text)
        cleaned.append((filename, text))
    return cleaned

def split_text(text):
    """Разбивает текст на контексты (абзацы)"""
    contexts = text.split('\n')
    contexts = [c.strip() + '\n' for c in contexts if c.strip()]
    return contexts

def get_names_in_text(text):
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    doc.parse_syntax(syntax_parser)
    doc.tag_ner(ner_tagger)
    for span in doc.spans:
        span.normalize(morph_vocab)
    for span in doc.spans:
        if span.type == PER:
            span.extract_fact(names_extractor)

    res = []
    for span in doc.spans:
        if span.fact:
            cur_elem = {
                 'normal': span.normal,
                 'start': span.start,
                 'end': span.stop,
                 'first': span.fact.as_dict.get('first'),
                 'middle': span.fact.as_dict.get('middle'),
                 'last': span.fact.as_dict.get('last')
            }
            res.append(cur_elem)
    return res

def get_names_in_texts(texts):
    """Обрабатывает все тексты из корпуса"""
    results = {}
    for filename, text in texts:
        contexts = split_text(text)
        file_res = []
        for context in contexts:
            file_res.extend(get_names_in_text(context))
        results[filename] = file_res
    return results

def save_names_to_csv(names_dict, save_path, sep='|'):
    rows = []
    for fname, names in names_dict.items():
        for n in names:
            n_row = {'filename': fname}
            n_row.update(n)
            rows.append(n_row)
    df = pd.DataFrame(rows)
    df.to_csv(save_path, sep=sep, index=False)

In [7]:
# Стратегия извлечения уникального имени
def extract_uniquely(names):
    res = []
    for name in names:
        cur_name = {'start': name['start'], 'end': name['end']}
        if name.get('last'):
            cur_name['name'] = name['last']
        elif name.get('first'):
            cur_name['name'] = name['first']
        elif name.get('middle'):
            cur_name['name'] = name['middle']
        if 'name' in cur_name:
            res.append(cur_name)
    return res

In [8]:
# --- Запуск обработки ---
texts = open_all_texts(corpus_path)
texts = preprocess_texts(texts)
all_names = get_names_in_texts(texts)

In [9]:
# Сохраняем и читаем CSV
save_names_to_csv(all_names, DATA_PATH / 'names.csv')
names_df = pd.read_csv(DATA_PATH / 'names.csv', sep='|')
names_df.fillna('?', inplace=True)
all_names_list = list(names_df.T.to_dict().values())

In [13]:
names = pd.read_csv(DATA_PATH / 'names.csv', sep='|')
names.head()

,filename,normal,start,end,first,middle,last
0,V401_Panikhida.txt,Андрей Андреич,167,181,Андрей,NaN,Андреич
1,V401_Panikhida.txt,Матвей,101,107,Матвей,NaN,NaN
2,V401_Panikhida.txt,Лопухов,190,198,NaN,NaN,Лопухов
3,V401_Panikhida.txt,Григорий,413,421,Григорий,NaN,NaN
4,V401_Panikhida.txt,Андрей Андреич,0,14,Андрей,NaN,Андреич


In [ ]:
# Извлечение уникальных имен
unique_names = extract_uniquely(all_names_list)

In [12]:
unique_names = extract_uniquely(all_names_list)
unique_names[:5]

[{'start': 167, 'end': 181, 'name': 'Андреич'},
 {'start': 101, 'end': 107, 'name': '?'},
 {'start': 190, 'end': 198, 'name': 'Лопухов'},
 {'start': 413, 'end': 421, 'name': '?'},
 {'start': 0, 'end': 14, 'name': 'Андреич'}]

In [14]:
# --- Класс для хранения матрицы совместного употребления ---
class GraphMatrix:
    """
    matrix - матрица совместного употребления (имена x имена)
    name_cnt - частотный словарь имен
    names - список имен
    """
    def __init__(self, matrix, name_cnt, names):
        self.matrix = matrix
        self.name_cnt = name_cnt
        self.names = names

    def update_by_ids(self, new_ids):
        """Обновление матрицы и словаря имен по выбранным индексам"""
        self.matrix = np.take(self.matrix, new_ids, axis=0)
        self.matrix = np.take(self.matrix, new_ids, axis=1)
        self.names = [self.names[i] for i in new_ids]
        self.name_cnt = {name: self.name_cnt[name] for name in self.names}

    def matrix_to_dict(self):
        """Преобразование матрицы в словарь для networkx"""
        res = {name:{} for name in self.names}
        for i in range(len(self.names)):
            for j in range(len(self.names)):
                if int(self.matrix[i,j]) > 0:
                    res[self.names[i]][self.names[j]] = {'weight': int(self.matrix[i,j])}
        return res

In [15]:
# --- Функции для построения матрицы совместного употребления ---
def update_matrix_cnt(matrix, name_to_id, context_names):
    """Обновление матрицы совместной встречаемости для одного контекста"""
    for i in range(len(context_names)-1):
        for j in range(i+1, len(context_names)):
            n1 = name_to_id[context_names[i]]
            n2 = name_to_id[context_names[j]]
            matrix[n1][n2] += 1
            matrix[n2][n1] += 1

def build_matrix_cnt(contexts, unique_names):
    """Строит матрицу совместной встречаемости по контекстам"""
    names = list({name['name'] for name in unique_names})
    name_cnt = {name:0 for name in names}
    for n in unique_names:
        name_cnt[n['name']] += 1

    matrix = np.zeros((len(names), len(names)), dtype=int)
    name_to_id = {names[i]: i for i in range(len(names))}

    context_start = 0
    name_index = 0

    for context in contexts:
        context_names = set()
        while name_index < len(unique_names):
            n = unique_names[name_index]
            if n['start'] >= context_start + len(context):
                break
            context_names.add(n['name'])
            name_index += 1
        update_matrix_cnt(matrix, name_to_id, list(context_names))
        context_start += len(context)

    return GraphMatrix(matrix, name_cnt, names)

In [16]:
# --- Фильтрация по частоте и совместной встречаемости ---
def filter_gapaxes_cnt(graph_matrix, min_count):
    new_ids = [i for i, name in enumerate(graph_matrix.names) if graph_matrix.name_cnt[name] >= min_count]
    graph_matrix.update_by_ids(new_ids)
    return graph_matrix

def filter_gapaxes_together(graph_matrix, min_weight):
    new_ids = [i for i in range(len(graph_matrix.names)) if graph_matrix.matrix[i].sum() >= min_weight]
    graph_matrix.update_by_ids(new_ids)
    return graph_matrix

In [17]:
# --- Построение графа ---
def save_graph(graph_matrix, save_path):
    G = nx.from_dict_of_dicts(graph_matrix.matrix_to_dict())
    nx.set_node_attributes(G, graph_matrix.name_cnt, 'weight')
    nx.write_gexf(G, save_path)

In [18]:
# --- Построение графа по контекстам ---
# Для примера возьмем один текст
example_filename, example_text = texts[0]
contexts = split_text(example_text)
graph_matrix = build_matrix_cnt(contexts, unique_names)

# Фильтрация имен
graph_matrix = filter_gapaxes_cnt(graph_matrix, 2)
graph_matrix = filter_gapaxes_together(graph_matrix, 1)

# Сохраняем граф для Gephi
save_graph(graph_matrix, DATA_PATH / 'mm_names_cnt.gexf')

print('Граф сохранён, размер матрицы:', graph_matrix.matrix.shape)

Граф сохранён, размер матрицы: (158, 158)


In [19]:
# --- Построение графа на основе эмбеддингов (совместная встречаемость через контексты) ---
def get_weight(vector1, vector2):
    return (1 - cosine(vector1, vector2)) * 100

def build_by_emb_matrix(emb_matrix):
    res = np.zeros((emb_matrix.shape[0], emb_matrix.shape[0]))
    for i in range(emb_matrix.shape[0]-1):
        for j in range(i+1, emb_matrix.shape[0]):
            res[i,j] = get_weight(emb_matrix[i], emb_matrix[j])
            res[j,i] = res[i,j]
    return res

def build_matrix_similarity(contexts, unique_names):
    names = list({n['name'] for n in unique_names})
    name_cnt = {name:0 for name in names}
    for n in unique_names:
        name_cnt[n['name']] += 1

    emb_matrix = np.zeros((len(names), len(contexts)))
    name_to_id = {names[i]: i for i in range(len(names))}

    context_start = 0
    name_index = 0

    for ci, context in enumerate(contexts):
        while name_index < len(unique_names):
            n = unique_names[name_index]
            if n['start'] >= context_start + len(context):
                break
            emb_matrix[name_to_id[n['name']], ci] += 1
            name_index += 1
        context_start += len(context)

    matrix = build_by_emb_matrix(emb_matrix)
    return GraphMatrix(matrix, name_cnt, names)

graph_matrix_similarity = build_matrix_similarity(contexts, unique_names)
graph_matrix_similarity = filter_gapaxes_cnt(graph_matrix_similarity, 2)
graph_matrix_similarity = filter_gapaxes_together(graph_matrix_similarity, 1)
save_graph(graph_matrix_similarity, DATA_PATH / 'mm_names_similarity.gexf')

print('Граф по эмбеддингам сохранён, размер матрицы:', graph_matrix_similarity.matrix.shape)

Граф по эмбеддингам сохранён, размер матрицы: (158, 158)
